# Refactoring Notebook Logic into Reusable Python Functions

This companion notebook turns the King County cleaning rules from notebook-style exploration into **small reusable functions**.

Its focus is narrower than the main `03` notebook:
- identify the stable transformation,
- wrap it in a function,
- keep the function deterministic and reusable,
- and verify the cleaned DataFrame still behaves as expected.


> **Checkpoint:**
> You can explain why each helper function should own one transformation responsibility.

> **Common pitfalls:**
> - Forgetting to assign `df = df.copy()` inside the function.
> - Leaving notebook-only inspection code inside reusable helpers.
> - Returning a DataFrame with extra temporary columns by accident.

> **Self-check:**
> Could you chain all four functions in a script without needing any manual cleanup in between?


### Loading data and refactor context

We reload the raw dataset here so the function-extraction workflow is self-contained. The raw rows are the same as in the main notebook; what changes is the **coding style** we use to express the cleanup logic.


In [ ]:
# pandas handles the tabular data work in this notebook.
import pandas as pd

In [ ]:
# Load the same raw dataset used in the main notebook.
# Here we focus specifically on turning notebook logic into reusable functions.
df = pd.read_csv("../data/King_County_House_prices_dataset.csv")

# Preview the raw rows before we start extracting functions.
df.head()

## Data Preparation

This notebook focuses on one specific task: taking notebook cleanup logic and rewriting it as **small reusable functions**.

The main notebook explains the exploratory reasoning. This notebook focuses on the implementation pattern that should end up in `src/king_county_refactoring`.


## Visual Guide: King County Cleaning Pipeline

```mermaid
flowchart TD
    A["Raw housing data"]
    B["Ratio outlier cleanup"]
    C["Basement normalization"]
    D["Create last_known_change"]
    E["Fill view and waterfront missings"]
    F["Cleaned DataFrame"]

    A --> B --> C --> D --> E --> F
```


#### Removing outliers:

The data scientist who performed the EDA noted suspicious bedroom/bathroom ratios but did not specify a cutoff.

In a real project, we would confirm this threshold with a domain expert. For this lesson, we turn the observed pattern into one explicit cleaning rule that can be reused in code.

> **Checkpoint:**
> You can explain why the cutoff is a modeling assumption, not an objective law.

> **Common pitfalls:**
> - Hard-coding one suspicious row instead of encoding the general rule.
> - Mutating the caller's DataFrame inside a helper function.


In [ ]:
# Create a helper feature so we can study the bathroom-to-bedroom relationship.
df["bath_bed_ratio"] = df["bathrooms"] / df["bedrooms"]

In [ ]:
# Summarize the helper feature before we decide on a filtering rule.
df["bath_bed_ratio"].describe()

In [ ]:
# Inspect the largest ratios first to see the extreme high-end outliers.
df["bath_bed_ratio"].sort_values(ascending=False).head(10)

In [ ]:
# Count how many rows violate the high-ratio threshold.
df.query("bath_bed_ratio >= 2")["id"].count()

In [ ]:
# Inspect the smallest ratios to see the extreme low-end outliers.
df["bath_bed_ratio"].sort_values(ascending=True).head(5)

We can see several properties with unusual bedroom/bathroom combinations. A strict rule for this exercise:
- drop rows where `bathrooms / bedrooms >= 2`,
- and drop rows where `bathrooms / bedrooms <= 0.10`.

This keeps the **filtering logic explicit and reusable** instead of tied to one single row index.


##### Original one-off notebook code

In the exploratory notebook phase, we might remove the suspicious record directly by index. The next cell demonstrates that notebook-style fix on a throwaway copy, so it can be run safely without changing the working DataFrame used later in the lesson.


In [ ]:
# Demonstrate the original one-off cleanup on a throwaway copy.
df_original_demo = df.copy()

# Only drop the row if it is still present in this copy.
if 15856 in df_original_demo.index:
    df_original_demo.drop(15856, axis=0, inplace=True)

# Show the updated shape after the notebook-style one-off fix.
df_original_demo.shape

In [ ]:
def bath_bed_ratio_outlier(df):
    # Copy first so the caller keeps the original DataFrame unchanged.
    df = df.copy()

    # Build a temporary helper feature that makes the filtering logic readable.
    df["bath_bed_ratio"] = df["bathrooms"] / df["bedrooms"]

    # Mark all rows whose ratio falls outside the accepted teaching range.
    invalid_ratio = (df["bath_bed_ratio"] >= 2) | (df["bath_bed_ratio"] <= 0.10)

    # Keep only valid rows, then remove the helper column before returning.
    df = df.loc[~invalid_ratio].copy()
    df.drop(columns=["bath_bed_ratio"], inplace=True)
    return df

In [ ]:
# Apply the reusable function to the working DataFrame.
df = bath_bed_ratio_outlier(df)

In [ ]:
# Recompute the ratio only as a temporary inspection series.
# This lets us check the cleaned distribution without changing the working schema.
(df["bathrooms"] / df["bedrooms"]).describe()

#### Dealing with string values in numeric columns:

The raw `sqft_basement` column contains a string placeholder (`?`) even though basement area should be numeric.

##### Cleanup strategy:
1. Trust the reliable source columns `sqft_living` and `sqft_above`.
2. Rebuild basement area directly as `sqft_living - sqft_above`.

This is a good candidate for a function because the transformation is deterministic and easy to reuse.

##### Original notebook-style code

The next cell shows the direct notebook-style cleanup that is useful during exploration. It runs on a temporary copy, so it can be compared with the reusable function that follows.


In [ ]:
# Demonstrate the basement rebuild on a temporary copy.
df_original_demo = df.copy()

# Recompute the feature from the two reliable source columns.
df_original_demo["sqft_basement"] = (
    df_original_demo["sqft_living"] - df_original_demo["sqft_above"]
)

# Inspect a few cleaned values.
df_original_demo[["sqft_living", "sqft_above", "sqft_basement"]].head()

In [ ]:
def sqft_basement(df):
    # Work on a copy so this helper is safe to reuse in a chain of functions.
    df = df.copy()

    # Recompute basement size from two more reliable source columns.
    df["sqft_basement"] = df["sqft_living"] - df["sqft_above"]
    return df

In [ ]:
# Apply the reusable basement-rebuild function.
df = sqft_basement(df)

Now we'll take care of the `yr_renovated` column by turning the notebook logic into a focused transformation function.

This step is a classic refactor target because it:
- combines two columns into one business-facing feature,
- contains logic we may want to reuse later,
- and should not depend on notebook-only state.

##### Original notebook-style code

The next cell keeps the original row-by-row style that often appears in notebooks. Again, it runs on a separate copy, so the example can be executed safely before moving to the reusable function.


In [ ]:
# Demonstrate the original year-consolidation logic on a temporary copy.
df_original_demo = df.copy()
last_known_change = []

# Build the consolidated year feature row by row.
for idx, yr_re in df_original_demo["yr_renovated"].items():
    if str(yr_re) == "nan" or yr_re == 0.0:
        last_known_change.append(df_original_demo["yr_built"][idx])
    else:
        last_known_change.append(int(yr_re))

# Add the derived column and remove the original source columns.
df_original_demo["last_known_change"] = last_known_change
df_original_demo.drop("yr_renovated", axis=1, inplace=True)
df_original_demo.drop("yr_built", axis=1, inplace=True)

# Show the relevant schema outcome.
df_original_demo[["last_known_change"]].head()

In [ ]:
def calculate_last_change(df):
    # Copy first so the original input DataFrame stays untouched.
    df = df.copy()

    # Build the new consolidated year feature row by row.
    last_known_change = []
    for idx, yr_re in df["yr_renovated"].items():
        # Missing or zero renovation years mean we fall back to the build year.
        if str(yr_re) == "nan" or yr_re == 0.0:
            last_known_change.append(df["yr_built"][idx])
        else:
            # Otherwise, keep the renovation year as the latest known change.
            last_known_change.append(int(yr_re))

    # Add the consolidated feature and drop the now-redundant source columns.
    df["last_known_change"] = last_known_change
    df.drop("yr_renovated", axis=1, inplace=True)
    df.drop("yr_built", axis=1, inplace=True)
    return df

In [ ]:
# Apply the consolidated-year transformation.
df = calculate_last_change(df)

In [ ]:
# Check the schema after the consolidation step.
df.info()

#### Dealing with missing values in `waterfront` and `view`:

Because both columns are mostly zeros in this dataset, we fill missing values with `0` for this exercise.

This function is intentionally small: one responsibility, one clear output, no notebook-only display logic.

##### Original notebook-style code

The next cell shows the direct fill approach exactly as it might appear during exploration, again on a throwaway copy so it stays separate from the reusable implementation.

> **Self-check:**
> Confirm that `view` and `waterfront` contain no missing values after this step.


In [ ]:
# Demonstrate the original missing-value fill logic on a temporary copy.
df_original_demo = df.copy()

# Fill both nullable visibility columns with the explicit business value 0.
df_original_demo["view"] = df_original_demo["view"].fillna(0)
df_original_demo["waterfront"] = df_original_demo["waterfront"].fillna(0)

# Verify that the two columns no longer contain missing values.
df_original_demo[["view", "waterfront"]].isna().sum()

In [ ]:
def fill_missings_view_wf(df):
    # Copy first so this fill step is safe to reuse anywhere.
    df = df.copy()

    # In this dataset, 0 is a meaningful explicit value for both columns.
    df["view"] = df["view"].fillna(0)
    df["waterfront"] = df["waterfront"].fillna(0)
    return df

In [ ]:
# Apply the final missing-value cleanup function.
df = fill_missings_view_wf(df)

In [ ]:
# Audit the remaining missing values after the full function chain.
missing_values = df.isnull().sum().to_frame(name="count")
missing_values["percentage"] = missing_values["count"] / df.shape[0] * 100
missing_values.query("count != 0")

### Visual Guide: Function Chain for the Final Cleanup

```mermaid
flowchart TD
    A["Raw DataFrame"]
    B["bath_bed_ratio_outlier()"]
    C["sqft_basement()"]
    D["calculate_last_change()"]
    E["fill_missings_view_wf()"]
    F["Cleaned DataFrame"]

    A --> B --> C --> D --> E --> F
```


Now convert this cleaned workflow into reusable Python functions.

This notebook is the implementation companion to `03-from-jupyter-notebook-to-python-scripts-example.ipynb`: the main notebook explains the reasoning, and this one shows how each notebook pattern becomes a reusable function.

**Exercise goal:** Finalize the notebook refactor into a starter Python module and validate the transformed DataFrame.

@TODO:
1. Implement the four data preparation functions in `src/king_county_refactoring/data_preparation.py`.
2. Keep each function DataFrame-in/DataFrame-out and copy-first.
3. Chain the functions in the same order used here.
4. Confirm that `view` and `waterfront` have no missing values after the chain.

**Hints:**
- Keep transformations deterministic.
- Avoid mutating caller-owned DataFrames.
- Remove temporary helper columns before returning.

Starter file: `src/king_county_refactoring/data_preparation.py`  
Reference solution: `src/king_county_refactoring/data_preparation_solution.py`
